In [16]:
#Load android feature selected dataset for modeling:
import pandas as pd
android_selected_features = pd.read_csv("../Processed_Data/Selected_FeaturesDatasets/android_SelectedFeatures.csv")

In [17]:
#Preparing dataset for training:
#Select relevant columns for analysis:
y = android_selected_features['stress']
X = android_selected_features.drop(columns=['stress', 'uid', 'day'])  # Drop target, ID columns, and date
print("Final feature set columns:", X.columns)
print("Final feature set shape:", X.shape)  

Final feature set columns: Index(['Unnamed: 0', 'race_american indian/alaska native',
       'audio_amp_mean_ep_2', 'loc_food_dur', 'call_out_num_ep_2',
       'race_american indian/white', 'light_mean_ep_3',
       'loc_max_dis_from_campus_ep_0', 'loc_study_dur', 'act_in_vehicle_ep_0',
       'phq4-4', 'pam', 'call_in_num_ep_0', 'act_in_vehicle_ep_2',
       'loc_social_dur', 'gender', 'sse3-4', 'phq4-1', 'race_asian',
       'light_mean_ep_1', 'quality_activity', 'loc_self_dorm_dur',
       'loc_self_dorm_audio_voice', 'loc_self_dorm_audio_amp',
       'audio_amp_std_ep_0', 'race_white', 'race_other/hispanic',
       'sms_in_num_ep_0', 'race_more than one', 'quality_loc',
       'act_on_foot_ep_1', 'loc_social_unlock_num', 'phq4-2', 'act_still_ep_3',
       'loc_social_convo_duration', 'audio_amp_std_ep_2',
       'audio_amp_mean_ep_3', 'sse3-1', 'sse3-3', 'race_black',
       'race_alaskan native/white', 'loc_other_dorm_audio_voice', 'phq4_score',
       'act_in_vehicle_ep_3', 'sse3

In [18]:
#Splitting data into test and train sets to prevent data leakage:
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold

#Column that identifies groups (participants):
group_col = 'uid'

#80/20 training testing split, with one testing group:
gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=android_selected_features[group_col]))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = android_selected_features[group_col].iloc[train_idx]

#Checking stress label distribution to ensure the groups are stratified:
print("Train distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True))

Train distribution:
stress
2.0    0.345873
3.0    0.253521
1.0    0.253165
4.0    0.110537
5.0    0.036905
Name: proportion, dtype: float64

Test distribution:
stress
3.0    0.295689
2.0    0.282332
1.0    0.239830
4.0    0.100182
5.0    0.081967
Name: proportion, dtype: float64


In [19]:
#Stratified group k-fold cross-validation to evaluate model performance:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_train, y_train, groups=groups_train)):
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

In [34]:
#Random forest regression model with stratifed group k-fold cross-validation:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score    


rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt', 
    random_state=42,
    n_jobs=-1)
RF_fold_mse = []
RF_fold_r2 = []

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_train, y_train, groups=groups_train)):
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    rf_model.fit(X_fold_train, y_fold_train)
    val_preds = rf_model.predict(X_fold_val)
    
    mse = mean_squared_error(y_fold_val, val_preds)
    r2 = r2_score(y_fold_val, val_preds)
    RF_fold_mse.append(mse)
    RF_fold_r2.append(r2)
    print(f"Fold {fold+1} - MSE: {mse:.4f}, R^2: {r2:.4f}")

    
print(f"\nAverage MSE across folds: {sum(RF_fold_mse)/len(RF_fold_mse):.4f}")
print(f"Average R^2 across folds: {sum(RF_fold_r2)/len(RF_fold_r2):.4f}") 

Fold 1 - MSE: 0.9180, R^2: 0.3354
Fold 2 - MSE: 0.8174, R^2: 0.2480
Fold 3 - MSE: 0.6717, R^2: 0.4401
Fold 4 - MSE: 0.5275, R^2: 0.4742
Fold 5 - MSE: 0.7184, R^2: 0.3755

Average MSE across folds: 0.7306
Average R^2 across folds: 0.3746


In [33]:
#XGBoost regression model with stratifed group k-fold cross-validation:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=10,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1)

XGB_fold_mse = []
XGB_fold_r2 = []

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_train, y_train, groups=groups_train)):
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    xgb_model.fit(X_fold_train, y_fold_train)
    val_preds = xgb_model.predict(X_fold_val)
    
    mse = mean_squared_error(y_fold_val, val_preds)
    r2 = r2_score(y_fold_val, val_preds)
    XGB_fold_mse.append(mse)
    XGB_fold_r2.append(r2)
    print(f"Fold {fold+1} - MSE: {mse:.4f}, R^2: {r2:.4f}")

print(f"\nAverage MSE across folds: {sum(XGB_fold_mse)/len(XGB_fold_mse):.4f}")
print(f"Average R^2 across folds: {sum(XGB_fold_r2)/len(XGB_fold_r2):.4f}")

Fold 1 - MSE: 0.9686, R^2: 0.2987
Fold 2 - MSE: 0.8393, R^2: 0.2278
Fold 3 - MSE: 0.6963, R^2: 0.4195
Fold 4 - MSE: 0.5536, R^2: 0.4482
Fold 5 - MSE: 0.7102, R^2: 0.3826

Average MSE across folds: 0.7536
Average R^2 across folds: 0.3554


**Personalized Models**

In [41]:
from sklearn.model_selection import train_test_split

#Creating personalized models for each participant:
unique_participants = android_selected_features['uid'].unique()
personalized_results = {}

for participant in unique_participants:
    participant_data = android_selected_features[android_selected_features['uid'] == participant]
    X_participant = participant_data.drop(columns=['stress', 'uid', 'day'])
    y_participant = participant_data['stress']
    
    if len(participant_data) < 10:  # Skip participants with too few samples
        continue
    
    X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
        X_participant, y_participant, test_size=0.2, random_state=42)
    
    model = XGBRegressor(
        n_estimators=200,
        max_depth=10,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1)
    
    model.fit(X_train_p, y_train_p)
    preds = model.predict(X_test_p)
    
    mse = mean_squared_error(y_test_p, preds)
    r2 = r2_score(y_test_p, preds)
    
    personalized_results[participant] = {'mse': mse, 'r2': r2}


#Average error across personalized models:
avg_mse = sum(result['mse'] for result in personalized_results.values()) / len(personalized_results)
avg_r2 = sum(result['r2'] for result in personalized_results.values()) / len(personalized_results)
print(f"\nAverage MSE across personalized models: {avg_mse:.4f}")
print(f"Average R^2 across personalized models: {avg_r2:.4f}")


Average MSE across personalized models: 0.5585
Average R^2 across personalized models: 0.3047


In [40]:
#personalized models with random forest regressor:
personalized_rf_results = {}

for participant in unique_participants:
    participant_data = android_selected_features[android_selected_features['uid'] == participant]
    X_participant = participant_data.drop(columns=['stress', 'uid', 'day'])
    y_participant = participant_data['stress']
    
    if len(participant_data) < 10:  # Skip participants with too few samples
        continue
    
    X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
        X_participant, y_participant, test_size=0.2, random_state=42)
    
    rf_model = RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features='sqrt', 
        random_state=42,
        n_jobs=-1)
    
    rf_model.fit(X_train_p, y_train_p)
    preds = rf_model.predict(X_test_p)
    
    mse = mean_squared_error(y_test_p, preds)
    r2 = r2_score(y_test_p, preds)
    
    personalized_rf_results[participant] = {'mse': mse, 'r2': r2}

#Average error across personalized random forest models:
avg_rf_mse = sum(result['mse'] for result in personalized_rf_results.values()) / len(personalized_rf_results)
avg_rf_r2 = sum(result['r2'] for result in personalized_rf_results.values()) / len(personalized_rf_results)
print(f"\nAverage RF MSE across personalized models: {avg_rf_mse:.4f}")
print(f"Average RF R^2 across personalized models: {avg_rf_r2:.4f}")


Average RF MSE across personalized models: 0.5883
Average RF R^2 across personalized models: 0.2879
